# 28. Pipeline Parallelism MicroBatch | Pipeline 并行微批次

**难度：** Hard | **环境：** CPU-first | **标签：** `并行通信`, `Pipeline Parallelism`, `MicroBatch` | **目标人群：** 并行通信学习者

---

## 本节导读

当完整模型无法放入一张 GPU 时，可以把连续的模型层分配给多个 stage，让不同设备分别承担不同片段的计算。真正的难点在于：如果每个 batch 都要从第一个 stage 依次走到最后一个 stage，stage 之间会频繁等待，设备利用率就会被流水线气泡拉低。

本节从一个可画清楚的灌入—排空时间轴开始，观察 micro-batch 如何让多个 stage 交错工作，再用 stage 数和 micro-batch 数分析气泡比例、激活保存和调度代价。学习完后，你应能解释为什么 micro-batch 不是越多越好，以及为什么 stage 切分、负载均衡和通信路径必须一起考虑。

**关键词：** `Pipeline Parallelism`, `Micro-batch`, `stage`, `bubble ratio`, `负载均衡`

---


## 前置阅读

**导语：** 进入本节前，先能根据模型层和设备之间的连接关系选择并行方式，再观察 micro-batch 如何让不同 stage 交错执行。
- [P1: 26. Parallel Strategy Decision Framework | 并行策略决策框架](../01_Hardware_Math_and_Systems/26_Parallel_Strategy_Decision_Framework.ipynb)
- [P1: 05. Communication Topologies | 通信拓扑与分布式基石](../01_Hardware_Math_and_Systems/05_Communication_Topologies.ipynb)

---

### Step 1：理解 micro-batch 如何填充流水线

Pipeline 并行先把模型层划分为多个 stage，再把一个大 batch 切成 $m$ 个 micro-batch。一个 micro-batch 完成当前 stage 后立即交给下一个 stage，当前 stage 则可以接着处理下一个 micro-batch；这样不同 stage 才有机会同时工作。

观察这条时间轴时，可以把每个格子读成“某个 stage 在某个时间步处理哪个 micro-batch”。开始时，后面的 stage 尚未收到数据；结束时，前面的 stage 已经没有新的 micro-batch。两段没有被计算占用的区域就是流水线气泡。增加 micro-batch 会让中间的连续工作区变长，但也会增加激活保存和调度事件。

这里的时间轴只记录前向占用，用来计算 stage、micro-batch 和 bubble ratio 的关系；真实训练还会在同一套 stage 上交错前向与反向计算。

![Pipeline 并行与微批次总览](../docs/public/02_PyTorch_Algorithms/28_pipeline_overview.svg)


### Step 2：从时间轴推导 bubble ratio

设 $p$ 为 Pipeline stage 数，$m$ 为全局 batch 被切出的 micro-batch 数。每个 micro-batch 要经过全部 $p$ 个 stage，因此实际完成的活跃槽位为 $m \times p$；从第一个 stage 开始到最后一个 stage 完成，时间轴长度为 $m + p - 1$，全部 stage 在这些时间步中的可能槽位为 $p \times (m + p - 1)$。

灌入和排空阶段没有被计算占用的槽位就是 bubble。于是：

$\text{Bubble Ratio} = \frac{p \times (m + p - 1) - m \times p}{p \times (m + p - 1)} = \frac{p - 1}{m + p - 1}$。

当 $m$ 远大于 $p$ 时，比例近似为 $(p-1)/m$；但这个理想公式没有计入 stage 不均衡和通信等待，真实 step time 仍要看最慢 stage 和通信是否落在关键路径上。

![简化流水线中 micro-batch 越过 stage 的时间轴](../docs/public/02_PyTorch_Algorithms/28_pipeline_bubble.svg)


### Step 3：在气泡、显存与调度开销间取舍

理想气泡率只描述时间轴上的空槽位。实际选择 micro-batch 时，还要同时观察每个 stage 的计算时间、边界通信和激活保存量：气泡变小并不保证端到端吞吐一定提高。

| 变化 | bubble 与设备利用率 | 显存与调度代价 | 下一步判断 |
|---|---|---|---|
| 增加 micro-batch | 灌入、排空占比通常下降 | 激活片段和调度事件增加 | 先检查单卡显存与激活生命周期 |
| 增加 stage 数 | 单卡层数下降，但灌入更慢 | stage 边界通信和气泡可能上升 | 检查层切分是否均衡 |
| stage 耗时不均 | 快 stage 等待慢 stage | 吞吐受最慢 stage 限制 | 重新切分层或调整 micro-batch |
| 通信落在关键路径 | 理想公式低估真实等待 | step time 上升 | 分开记录计算、通信和等待时间 |


### Step 4：CPU 实现——构建时间轴并验证气泡率

下面把 Step1–3 的时间轴和公式落实为两个 CPU 机制函数：`build_pipeline_timeline` 负责生成每个时间步的活跃 `(stage, micro_batch)`，`compute_bubble_ratio` 负责统计活跃槽位并计算气泡率。实现只围绕这两个不变量展开，测试区再检查覆盖次数、边界输入和公式结果。

| 函数 | 学习者完成的机制 | 必须满足的约束 | 测试证据 |
|---|---|---|---|
| `build_pipeline_timeline` | 构造灌入—排空时间轴 | 时间步为 `p + m - 1`；每个 `(stage, micro_batch)` 出现在 `t = stage + micro_batch` | 时间轴长度、坐标范围、推进顺序和覆盖次数 |
| `compute_bubble_ratio` | 从时间轴统计活跃槽位并计算 bubble | `active_slots = p × m`；非法 `p/m` 必须报错 | 精确公式对照、micro-batch 增加时的边界关系 |


In [ ]:
import torch

In [ ]:

def build_pipeline_timeline(p: int, m: int) -> list[list[tuple[int, int]]]:
    """
    构造一个简化的流水线时间轴，并保留 stage/micro-batch 的占用契约。

    `timeline[t]` 记录第 t 个时间步里活跃的 `(stage, micro_batch)`。
    每个 micro-batch 应恰好经过全部 `p` 个 stage，总活跃槽位为 `p * m`。

    Args:
        p: Pipeline stage 数量，必须为正整数。
        m: Micro-batch 数量，必须为正整数。
    """
    # 输入契约：p 和 m 必须为正整数，非法配置应明确抛出 ValueError。
    # ==========================================
    # TODO 1：构造 Pipeline 的时间轴
    # 变量提示（每个变量各占一行）：
    # p, m = int(p), int(m)
    # total_steps = p + m - 1
    # timeline = []
    # t = ...
    # stage = ...
    # active = []
    # micro_idx = t - stage
    # ==========================================
    pass


def compute_bubble_ratio(p: int, m: int) -> float:
    """
    根据时间轴计算流水线并行的气泡率。

    先用时间轴统计实际活跃槽位，再用所有 stage 在所有时间步都满载时
    的槽位数作为分母，避免把 bubble ratio 写成只依赖 `p`、`m` 的孤立公式。
    
    Args:
        p: Pipeline Stage 数量 (GPU 数量)
        m: Micro-batch 的数量
        
    Returns:
        float: 气泡占比 [0, 1]
    """
    # 输入契约：p 和 m 必须为正整数，避免空时间轴或零分母。
    # ==========================================
    # TODO 2：基于时间轴统计活跃槽位，并计算 Bubble Ratio
    # 变量提示（每个变量各占一行）：
    # timeline = build_pipeline_timeline(p, m)
    # active_slots = sum(len(step) for step in timeline)
    # total_slots = len(timeline) * p
    # bubble = 1 - active_slots / total_slots
    # ==========================================
    pass



In [ ]:
def test_pipeline_bubble():
    """验证时间轴活跃槽位与 bubble ratio 的数量关系。"""
    try:
        # 用固定 p/m 检查实现是否符合时间轴推导出的精确公式。
        ratio = compute_bubble_ratio(p=8, m=32)
        expected = (8 - 1) / (32 + 8 - 1)
        assert abs(ratio - expected) < 1e-12, f"Bubble Ratio 应为 {expected}，实际为 {ratio}"
        
        print("✅ 测试通过！在大规模集群训练时，为了降低流水线气泡，Micro-batch 的数量 m 必须远远大于 Stage 的数量 p。")
    except NotImplementedError:
        print("请先完成 TODO 部分的代码！")
        raise
    except (AttributeError, NameError, TypeError, ValueError, AssertionError, RuntimeError) as e:
        if isinstance(e, AttributeError):
            print("代码未完成，无法找到必要的属性")
        elif isinstance(e, NameError):
            print("代码可能未完成，导致了变量未定义")
        elif isinstance(e, TypeError):
            print("代码可能未完成，导致了类型错误")
        elif isinstance(e, ValueError):
            print("代码可能未完成，导致了张量维度错误")
        elif isinstance(e, AssertionError):
            print("代码可能未完成，导致了断言失败")
        else:
            print("代码可能未完成，导致了运行时错误")
        raise NotImplementedError("请先完成 TODO 部分的代码！") from e
    except Exception as e:
        print(f"❌ 测试失败: {e}")
        raise

def test_pipeline_timeline_structure():
    """验证时间轴长度、坐标范围以及每个 stage/micro-batch 的覆盖次数。"""
    timeline = build_pipeline_timeline(p=3, m=5)
    assert len(timeline) == 7
    assert all(0 <= stage < 3 and 0 <= micro < 5 for step in timeline for stage, micro in step)
    pairs = [pair for step in timeline for pair in step]
    assert len(pairs) == len(set(pairs)) == 15
    assert all(sum(micro == i for stage, micro in pairs) == 3 for i in range(5))
    assert all(sum(stage == i for stage, micro in pairs) == 5 for i in range(3))

def test_pipeline_active_slots():
    """验证活跃槽位总量等于 micro-batch 与 stage 的笛卡尔积。"""
    timeline = build_pipeline_timeline(p=3, m=5)
    assert sum(len(step) for step in timeline) == 15

def test_pipeline_progression_order():
    """验证每个 micro-batch 按 stage 顺序前进，每次只跨过一个 stage。"""
    timeline = build_pipeline_timeline(p=4, m=3)
    positions = {}
    for t, step in enumerate(timeline):
        for stage, micro in step:
            assert (stage, micro) not in positions
            assert t == stage + micro
            positions[(stage, micro)] = t
    for micro in range(3):
        assert [positions[(stage, micro)] for stage in range(4)] == [micro + stage for stage in range(4)]

def test_pipeline_bubble_boundary():
    """验证增加 micro-batch 不会放大理想流水线的 bubble ratio。"""
    assert compute_bubble_ratio(3, 8) <= compute_bubble_ratio(3, 4)

def test_pipeline_input_contract():
    """验证非法 stage 或 micro-batch 配置不会静默产生错误结果。"""
    for p, m in [(0, 4), (3, 0), (-1, 4)]:
        try:
            build_pipeline_timeline(p, m)
            raise AssertionError("非法配置应抛出 ValueError")
        except ValueError:
            pass
        try:
            compute_bubble_ratio(p, m)
            raise AssertionError("非法配置应抛出 ValueError")
        except ValueError:
            pass

def run_pipeline_test():
    """统一执行结构、槽位、边界和输入契约测试。"""
    try:
        test_pipeline_timeline_structure()
        test_pipeline_active_slots()
        test_pipeline_progression_order()
        test_pipeline_bubble_boundary()
        test_pipeline_input_contract()
        test_pipeline_bubble()
    except NotImplementedError:
        print("请先完成 TODO 部分的代码！")
        raise
    except Exception as e:
        print(f"❌ Pipeline 机制测试失败: {e}")
        raise

run_pipeline_test()



---

🛑 **STOP HERE** 🛑
<br><br><br><br><br><br><br><br><br><br>
> 请先尝试自己完成代码并跑通测试。<br>
> 如果你正在 Colab 中运行，并且遇到困难没有思路，可以向下滚动查看参考答案。
<br><br><br><br><br><br><br><br><br><br>

---


## 参考代码与解析

### 代码

In [ ]:
def build_pipeline_timeline(p: int, m: int) -> list[list[tuple[int, int]]]:
    if p <= 0 or m <= 0:
        raise ValueError("p 和 m 必须为正数")
    # TODO 1: 构造 Pipeline 的时间轴
    timeline = []
    total_steps = p + m - 1
    for t in range(total_steps):
        active = []
        for stage in range(p):
            micro_idx = t - stage
            if 0 <= micro_idx < m:
                active.append((stage, micro_idx))
        timeline.append(active)
    return timeline


def compute_bubble_ratio(p: int, m: int) -> float:
    if p <= 0 or m <= 0:
        raise ValueError("p 和 m 必须为正数")
    # TODO 2: 基于时间轴统计活跃槽位并计算 Bubble Ratio
    timeline = build_pipeline_timeline(p, m)
    active_slots = sum(len(step) for step in timeline)
    total_slots = len(timeline) * p
    bubble = 1 - active_slots / total_slots
    return bubble



### 解析

**1. TODO 1（构造时间轴）**
- 通过 `timeline[t]` 记录每个时间步里哪些 stage 正在处理哪些 micro-batch。
- `micro_idx = t - stage` 体现了简化流水线填充—排空阶段的对角线式占用关系；它不是完整的 forward/backward 1F1B 实现。
- 只要 `0 <= micro_idx < m`，就说明该 stage 在这个时间步是活跃的。

**2. TODO 2（统计活跃槽位并计算 Bubble Ratio）**
- `active_slots` 统计时间轴里所有活跃的 stage/micro-batch 组合数。
- `total_slots = len(timeline) * p` 表示如果每个时间步都满载时的槽位总数。
- `bubble = 1 - active_slots / total_slots` 就是气泡占比。

**3. 进阶思考**
- 当 `m` 远大于 `p` 时，`Bubble Ratio` 会快速下降。
- 这正是大规模训练里要尽量使用更多 micro-batch 的原因。
- 实际系统里还会叠加通信、重计算和显存约束，调度会更复杂。


### Step 5（可选）：GPU Pipeline Parallel benchmark

下面按 5.1–5.3 依次完成环境配置、真实执行和结果解释；GPU 实验默认关闭。


#### 5.1 环境与固定 workload

至少使用两张 GPU，固定模型分段、global batch、dtype、序列长度、warmup 与 repeats；G0 为单卡 baseline，G1 为固定 stage 的流水线，G2 只改变 micro-batch 数。

In [ ]:
# 5.1：默认关闭；真实 Pipeline 命令必须写入 RESULT_PATH。
RUN_GPU_EXPERIMENT = False
WORLD_SIZE = 2
PIPELINE_STAGES = 2
MICRO_BATCHES = 8
RESULT_PATH = 'benchmarks/results/28_pipeline_parallel_benchmark.json'
BENCHMARK_COMMAND = None
print({'run': RUN_GPU_EXPERIMENT, 'stages': PIPELINE_STAGES, 'micro_batches': MICRO_BATCHES, 'result': RESULT_PATH})


#### 5.2 配置与执行

使用 DeepSpeed Pipeline 或兼容实现运行真实 schedule；默认关闭，失败时保留 failure。

In [ ]:
# 5.2：运行真实 pipeline schedule；默认关闭。
if RUN_GPU_EXPERIMENT:
    if not BENCHMARK_COMMAND:
        raise ValueError('请填写 Pipeline BENCHMARK_COMMAND。')
    import subprocess
    subprocess.run(BENCHMARK_COMMAND, check=True)
else:
    print('GPU benchmark 默认关闭。')


#### 5.3 读取结果、解释指标与形成决策

记录 stage、micro-batch、workload、peak memory、step time、throughput、bubble ratio、通信时间、evidence level、failure 与 decision；只有吞吐收益未被调度与通信开销抵消时才接受。

In [ ]:
# 5.3：只读取真实 benchmark JSON。
import json
from pathlib import Path
if Path(RESULT_PATH).exists():
    result = json.loads(Path(RESULT_PATH).read_text())
    required = {'workload', 'hardware', 'metrics', 'evidence_level', 'failure', 'decision'}
    missing = required - set(result)
    if missing:
        raise ValueError(f'结果 JSON 缺少字段：{sorted(missing)}')
    print(result)
else:
    print(f'尚无真实 Pipeline 结果：{RESULT_PATH}')


## 相关阅读

流水线并行可以继续从 GPipe / 1F1B 调度论文进入真实框架，再和 ZeRO、Tensor Parallelism 对照。

- [GPipe 原论文](https://arxiv.org/abs/1811.06965)
- [Megatron-LM 开源仓库](https://github.com/NVIDIA/Megatron-LM)
- [27. ZeRO 优化器模拟](../02_PyTorch_Algorithms/27_ZeRO_Optimizer_Sim.ipynb)
- [29. Tensor 并行模拟](../02_PyTorch_Algorithms/29_Tensor_Parallelism_Sim.ipynb)
- [P1: CUDA Stream 与异步执行](../01_Hardware_Math_and_Systems/17_CUDA_Stream_and_Asynchrony.ipynb)
- [60. LoRA 微调项目](../02_PyTorch_Algorithms/60_LoRA_Fine_Tuning_Project.ipynb)
